# Tutorial 5: Survival Analysis

This tutorial demonstrates OneEHR's survival analysis capabilities
including Cox PH models, DeepSurv, concordance index, and Kaplan-Meier visualization.

In [ ]:
import numpy as np
import torch

## Survival Models

OneEHR provides two deep survival models:
- **DeepSurv**: Cox PH deep network (Katzman et al., 2018)
- **DeepHit**: Discrete-time competing risks model (Lee et al., 2018)

In [ ]:
from oneehr.models.survival import DeepSurv, DeepHit, CoxPHLoss

# Create a DeepSurv model
model = DeepSurv(input_dim=10, hidden_dim=64, num_layers=2, dropout=0.1)
print(f"DeepSurv parameters: {sum(p.numel() for p in model.parameters()):,}")

# Forward pass
x = torch.randn(32, 5, 10)  # 32 patients, 5 time steps, 10 features
lengths = torch.randint(1, 6, (32,))
risk_scores = model(x, lengths)
print(f"Risk scores shape: {risk_scores.shape}")

In [ ]:
# Cox PH loss function
loss_fn = CoxPHLoss()
times = torch.rand(32) * 10  # observed times
events = torch.bernoulli(torch.ones(32) * 0.7)  # event indicator

loss = loss_fn(risk_scores.squeeze(), times, events)
print(f"Cox PH loss: {loss.item():.4f}")

## Survival Metrics

In [ ]:
from oneehr.eval.survival import survival_metrics, concordance_index

# Simulate survival data
rng = np.random.default_rng(42)
n = 100
true_risk = rng.random(n)
event_times = np.exp(-true_risk + rng.normal(0, 0.5, n))
event_observed = rng.binomial(1, 0.7, n).astype(float)
predicted_risk = true_risk + rng.normal(0, 0.2, n)  # noisy prediction

result = survival_metrics(event_times, predicted_risk, event_observed)
print("Survival Metrics:")
for k, v in result.metrics.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

## Kaplan-Meier Visualization

In [ ]:
from oneehr.visualization.kaplan_meier import plot_kaplan_meier

# Create two risk groups
groups = np.where(predicted_risk > np.median(predicted_risk), "High Risk", "Low Risk")

fig = plot_kaplan_meier(
    event_times=event_times,
    event_observed=event_observed,
    groups=groups,
    group_labels=["High Risk", "Low Risk"],
    ci=True,
    title="Kaplan-Meier Survival Curves by Risk Group",
    xlabel="Time (days)",
    style="nature",
)
fig

## Survival Task Configuration

To run a survival analysis experiment, set `task.kind = "survival"` in your TOML config:

```toml
[task]
kind = "survival"
prediction_mode = "patient"

[[models]]
name = "deepsurv"
[models.params]
hidden_dim = 128
num_layers = 2

[[models]]
name = "deephit"
[models.params]
num_time_bins = 20
```